In [4]:
import os
import json
import numpy as np
import pandas as pd
import shap
import lightgbm as lgb
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.cm as cm

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.cluster import DBSCAN
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from statsmodels.stats.proportion import proportions_ztest

from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.units import cm as rl_cm
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Image as RLImage,
    PageBreak, Table, TableStyle
)


def insightforge_dsmodule(
    file_path,
    sep=";",
    decimal=",",
    id_col="ID",
    target_score_col="score",
    threshold=10,
    min_profile_size=0.05,
    eps=0.25,
    output_path="IF_analytical_output"
):
    """
    FULL ANALYTICAL PIPELINE

    PARAMETERS
    ----------
    file_path         : input CSV file path
    sep               : column separator
    decimal           : decimal separator
    id_col            : unique identifier column name
    target_score_col  : model score column name
    threshold         : percentile threshold for HP / LP definition
    min_profile_size  : minimum cluster size as fraction of total rows (0-1)
    eps               : DBSCAN neighbourhood radius
    output_path       : folder where all outputs are saved

    OUTPUT FILES
    ------------
    shap_importance.json    global SHAP mean absolute impact per feature
    PROFILE_RULES.csv       decision-tree rules per leaf -> Profile_ID
    PROFILE_STATS.csv       aggregate statistics per Profile_ID
    ID_W_PROFILE.csv        row-level table: id, Profile_ID, HP, LP
    PROFILE_FEATURES.csv    decision-tree feature importances (non-zero only)
    oddsratio.csv           odds-ratio from multinomial logistic regression
                            (columns = feature names, rows = Profile_ID classes)
    InsightForge_Report.pdf visual report
    """

    os.makedirs(output_path, exist_ok=True)

    # =========================================================
    # 1. LOAD DATA
    # =========================================================
    df = pd.read_csv(file_path, sep=sep, decimal=decimal)

    # =========================================================
    # 2. HP / LP / TRICT LABELS
    # =========================================================
    p_low, p_high = np.percentile(df[target_score_col], [threshold, 100 - threshold])

    df["HP"]    = np.where(df[target_score_col] >= p_high, 1, 0)
    df["LP"]    = np.where(df[target_score_col] <= p_low,  1, 0)
    df["TRICT"] = np.where(
        df[target_score_col] <= p_low,  0,
        np.where(df[target_score_col] > p_high, 2, 1)
    )

    base_propensity = df["HP"].mean()

    # =========================================================
    # 3. FEATURE MATRIX  (shared by SHAP model, PCA, DBSCAN)
    # =========================================================
    _internal_cols = {id_col, target_score_col, "HP", "LP", "TRICT"}

    raw_num_cols = [
        c for c in df.select_dtypes(exclude=["object", "category"]).columns
        if c not in _internal_cols
    ]
    raw_cat_cols = [
        c for c in df.select_dtypes(include=["object", "category"]).columns
        if c not in _internal_cols
    ]

    X_num = df[raw_num_cols].fillna(df[raw_num_cols].mean())

    ohe_main = OneHotEncoder(sparse_output=False, drop="first", handle_unknown="ignore")
    if raw_cat_cols:
        X_cat_arr     = ohe_main.fit_transform(df[raw_cat_cols])
        ohe_cat_names = ohe_main.get_feature_names_out(raw_cat_cols).tolist()
        X_cat_df      = pd.DataFrame(X_cat_arr, columns=ohe_cat_names, index=df.index)
    else:
        X_cat_df      = pd.DataFrame(index=df.index)
        ohe_cat_names = []

    X                   = pd.concat([X_num, X_cat_df], axis=1)
    feature_names_model = list(X.columns)

    scaler   = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)

    # =========================================================
    # 4. SHAP MODEL  (best of LightGBM vs RandomForest on HP)
    # =========================================================
    Y_hp = df["HP"].values

    search_gb = RandomizedSearchCV(
        lgb.LGBMClassifier(verbose=-1),
        {"n_estimators": [50, 100]},
        cv=3, scoring="roc_auc", n_jobs=-1, random_state=42
    )
    search_rf = RandomizedSearchCV(
        RandomForestClassifier(),
        {"n_estimators": [50, 100]},
        cv=3, scoring="roc_auc", n_jobs=-1, random_state=42
    )
    search_gb.fit(X_scaled, Y_hp)
    search_rf.fit(X_scaled, Y_hp)

    best_model = (
        search_gb if search_gb.best_score_ >= search_rf.best_score_
        else search_rf
    ).best_estimator_
    best_model.fit(X_scaled, Y_hp)

    explainer   = shap.TreeExplainer(best_model)
    shap_values = explainer.shap_values(X_scaled)

    # robust extraction -> always (n_samples, n_features)
    if isinstance(shap_values, list):
        shap_array = shap_values[1] if len(shap_values) > 1 else shap_values[0]
    else:
        shap_array = shap_values
    if shap_array.ndim == 3:
        shap_array = shap_array[:, :, 1]

    mean_impact = np.abs(shap_array).mean(axis=0).reshape(-1)

    shap_output = sorted(
        [{"feature": feat, "mean_impact": float(mean_impact[i])}
         for i, feat in enumerate(feature_names_model)],
        key=lambda x: x["mean_impact"], reverse=True
    )
    with open(f"{output_path}/shap_importance.json", "w") as f:
        json.dump(shap_output, f, indent=2)

    # =========================================================
    # 5. PCA  (dimensionality reduction for DBSCAN)
    # =========================================================
    pca_cluster = PCA(n_components=0.71, random_state=42)
    X_pca       = pca_cluster.fit_transform(X_scaled)

    # =========================================================
    # 6. DBSCAN  ->  raw cluster assignment
    # =========================================================
    min_samples       = max(2, int(min_profile_size * len(X_pca)))
    db                = DBSCAN(eps=eps, min_samples=min_samples, metric="cosine")
    df["cluster"]     = db.fit_predict(X_pca)

    df["cluster_propensity"] = df.groupby("cluster")["HP"].transform("mean")

    # pre-filter: keep cluster only if propensity ratio > 1.2x base
    df["final_cluster"] = np.where(
        df["cluster_propensity"] / base_propensity > 1.2,
        df["cluster"].astype(str),
        "NotHP"
    )

    # =========================================================
    # 7. Z-TEST ON CLUSTERS  (proportions test: cluster HP vs rest)
    # =========================================================
    cluster_ztest_rows = []
    for cl in df["final_cluster"].unique():
        mask_cl = df["final_cluster"] == cl
        z_stat, p_val = proportions_ztest(
            count=[df.loc[mask_cl,  "HP"].sum(), df.loc[~mask_cl, "HP"].sum()],
            nobs =[int(mask_cl.sum()), int((~mask_cl).sum())]
        )
        cluster_ztest_rows.append({
            "final_cluster":   cl,
            "cluster_p_value": p_val,
            "cluster_z_stat":  z_stat
        })

    df = df.merge(pd.DataFrame(cluster_ztest_rows), on="final_cluster", how="left")

    df["final_cluster"] = np.where(
        (df["cluster_p_value"] <= 0.05) & (np.abs(df["cluster_z_stat"]) >= 1.96),
        df["final_cluster"],
        "NotHP"
    )

    # =========================================================
    # 8. DECISION TREE  ->  leaf segmentation
    # =========================================================
    _drop_dt = {
        id_col, target_score_col,
        "HP", "LP", "TRICT",
        "cluster", "cluster_propensity",
        "final_cluster", "cluster_p_value", "cluster_z_stat"
    }
    X_dt_df = df.drop(columns=list(_drop_dt), errors="ignore")

    dt_num_cols = X_dt_df.select_dtypes(exclude=["object", "category"]).columns.tolist()
    dt_cat_cols = X_dt_df.select_dtypes(include=["object", "category"]).columns.tolist()

    ohe_dt = OneHotEncoder(sparse_output=False, drop="first", handle_unknown="ignore")
    if dt_cat_cols:
        X_dt_cat_arr = ohe_dt.fit_transform(X_dt_df[dt_cat_cols])
        dt_cat_names = ohe_dt.get_feature_names_out(dt_cat_cols).tolist()
        X_dt_cat_df  = pd.DataFrame(X_dt_cat_arr, columns=dt_cat_names, index=df.index)
    else:
        X_dt_cat_df  = pd.DataFrame(index=df.index)
        dt_cat_names = []

    X_dt          = pd.concat([X_dt_df[dt_num_cols], X_dt_cat_df], axis=1)
    dt_feat_names = list(X_dt.columns)
    X_dt_arr      = X_dt.fillna(X_dt.mean()).values
    Y_dt          = df["final_cluster"].values

    dt = DecisionTreeClassifier(random_state=42)
    dt.fit(X_dt_arr, Y_dt)

    df["leaf_id"]         = dt.apply(X_dt_arr)
    df["leaf_propensity"] = df.groupby("leaf_id")["HP"].transform("mean")

    # =========================================================
    # 9. Z-TEST ON LEAVES  ->  Profile_ID validation
    #
    #    Each leaf gets a named Profile_ID only if the proportion
    #    z-test (leaf HP rate vs rest HP rate) satisfies:
    #      |z_stat| >= 1.96  AND  p_value <= 0.05
    #    Otherwise the leaf is labelled "Generic".
    # =========================================================
    min_leaf_size = int(min_profile_size * len(df))   # soglia assoluta in righe

    leaf_ztest_rows = []
    for leaf in df["leaf_id"].unique():
        mask_leaf  = df["leaf_id"] == leaf
        leaf_count = int(mask_leaf.sum())
        z_stat, p_val = proportions_ztest(
            count=[df.loc[mask_leaf,  "HP"].sum(), df.loc[~mask_leaf, "HP"].sum()],
            nobs =[leaf_count, int((~mask_leaf).sum())]
        )
        leaf_ztest_rows.append({
            "leaf_id":      leaf,
            "leaf_count":   leaf_count,
            "leaf_p_value": p_val,
            "leaf_z_stat":  z_stat
        })

    df = df.merge(pd.DataFrame(leaf_ztest_rows), on="leaf_id", how="left")

    # Un leaf diventa Profile_ID named solo se supera TUTTI e TRE i criteri:
    #   1. dimensione >= min_profile_size (soglia assoluta in righe)
    #   2. p_value <= 0.05
    #   3. |z_stat| >= 1.96
    df["Profile_ID"] = np.where(
        (df["leaf_count"]   >= min_leaf_size) &
        (df["leaf_p_value"] <= 0.05) &
        (np.abs(df["leaf_z_stat"]) >= 1.96),
        df["leaf_id"].astype(str),
        "Generic"
    )

    # =========================================================
    # 10. RULE EXTRACTION  (decision-tree path to each leaf)
    # =========================================================
    def extract_rules(tree_clf, feat_names):
        cl = tree_clf.tree_.children_left
        cr = tree_clf.tree_.children_right
        ft = tree_clf.tree_.feature
        th = tree_clf.tree_.threshold
        rules = {}

        def recurse(node, path):
            if cl[node] == cr[node]:   # leaf node
                rules[node] = " AND ".join(path) if path else "ROOT"
            else:
                fname = feat_names[ft[node]]
                recurse(cl[node], path + [f"{fname} <= {th[node]:.3f}"])
                recurse(cr[node], path + [f"{fname} > {th[node]:.3f}"])

        recurse(0, [])
        return rules

    rules_dict = extract_rules(dt, dt_feat_names)

    df_rules = (
        df[["leaf_id", "Profile_ID"]]
        .drop_duplicates()
        .assign(rule=lambda d: d["leaf_id"].map(rules_dict))
        [["leaf_id", "Profile_ID", "rule"]]
        .sort_values("Profile_ID")
        .reset_index(drop=True)
    )
    df_rules.to_csv(f"{output_path}/PROFILE_RULES.csv", index=False)

    # =========================================================
    # 11. FEATURE IMPORTANCES  (decision tree, non-zero only)
    # =========================================================
    df_importances = (
        pd.DataFrame({"feature": dt_feat_names,
                      "importance": dt.feature_importances_})
        .query("importance > 0")
        .sort_values("importance", ascending=False)
        .reset_index(drop=True)
    )
    df_importances.to_csv(f"{output_path}/PROFILE_FEATURES.csv", index=False)

    # =========================================================
    # 12. PROFILE STATS  (mean of all numerics per Profile_ID)
    # =========================================================
    df["profile_volume"] = df.groupby("Profile_ID")[id_col].transform("count")
    df["profile_pct"]    = df["profile_volume"] / len(df)

    _exclude_agg = {
        "Profile_ID", id_col,
        "cluster", "leaf_id", "final_cluster",
        "cluster_p_value", "cluster_z_stat",
        "leaf_p_value", "leaf_z_stat", "leaf_count"
    }
    agg_num_cols = [
        c for c in df.select_dtypes(include="number").columns
        if c not in _exclude_agg
    ]

    df_stats = (
        df.groupby("Profile_ID")[agg_num_cols]
        .mean()
        .reset_index()
    )
    df_stats.to_csv(f"{output_path}/PROFILE_STATS.csv", index=False)

    # =========================================================
    # 13. ID -> PROFILE TABLE
    # =========================================================
    df[[id_col, "Profile_ID", "HP", "LP"]].to_csv(
        f"{output_path}/ID_W_PROFILE.csv", index=False
    )

    # =========================================================
    # 14. ODDS RATIO  (multinomial logistic regression)
    # =========================================================
    _drop_lr = _drop_dt | {
        "Profile_ID",
        "cluster_propensity", "leaf_propensity",
        "leaf_p_value", "leaf_z_stat", "leaf_count",
        "cluster_p_value", "cluster_z_stat",
        "profile_volume", "profile_pct"
    }
    X_lr_df = df.drop(columns=list(_drop_lr), errors="ignore")

    lr_num_cols = X_lr_df.select_dtypes(include=["number"]).columns.tolist()
    lr_cat_cols = X_lr_df.select_dtypes(exclude=["number"]).columns.tolist()

    X_lr_num = (
        SimpleImputer(strategy="mean").fit_transform(X_lr_df[lr_num_cols])
        if lr_num_cols else np.empty((len(X_lr_df), 0))
    )
    if lr_cat_cols:
        ohe_lr       = OneHotEncoder(sparse_output=False, drop="first", handle_unknown="ignore")
        X_lr_cat     = ohe_lr.fit_transform(X_lr_df[lr_cat_cols].fillna("missing"))
        lr_cat_names = ohe_lr.get_feature_names_out(lr_cat_cols).tolist()
    else:
        X_lr_cat     = np.empty((len(X_lr_df), 0))
        lr_cat_names = []

    lr_all_names = lr_num_cols + lr_cat_names
    X_lr_final   = MinMaxScaler().fit_transform(np.hstack([X_lr_num, X_lr_cat]))
    Y_lr         = df["Profile_ID"].values

    log = LogisticRegression(max_iter=5000, random_state=42)
    log.fit(X_lr_final, Y_lr)

    df_or = pd.DataFrame(
        np.exp(log.coef_),
        columns=lr_all_names,
        index=log.classes_
    )
    df_or.index.name = "Profile_ID"
    df_or.reset_index(inplace=True)
    df_or.to_csv(f"{output_path}/oddsratio.csv", index=False)

    # =========================================================
    # 15. CHARTS  (temporary PNGs embedded into PDF)
    # =========================================================
    plot_paths    = []
    profile_order = (
        sorted([p for p in df["Profile_ID"].unique() if p != "Generic"]) + ["Generic"]
    )

    # --- 15a. SHAP bar chart (top 20) ---
    shap_df = pd.DataFrame(shap_output).head(20)
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(shap_df["feature"][::-1], shap_df["mean_impact"][::-1], color="#4C72B0")
    ax.set_xlabel("Mean |SHAP| value")
    ax.set_title("Global SHAP Feature Importance (top 20)")
    plt.tight_layout()
    p_shap = f"{output_path}/_shap_bar.png"
    fig.savefig(p_shap, dpi=150)
    plt.close(fig)
    plot_paths.append(("Global SHAP Feature Importance", p_shap))

    # --- 15b. PCA 2D scatter coloured by Profile_ID ---
    pca2d   = PCA(n_components=2, random_state=42)
    X_pca2d = pca2d.fit_transform(X_scaled)
    var_exp = pca2d.explained_variance_ratio_ * 100

    profiles_arr    = df["Profile_ID"].values
    unique_profiles = sorted(set(profiles_arr), key=lambda x: (x == "Generic", x))
    cmap_scatter    = cm.get_cmap("tab10", len(unique_profiles))
    profile_color   = {p: cmap_scatter(i) for i, p in enumerate(unique_profiles)}

    fig, ax = plt.subplots(figsize=(9, 6))
    for pid in unique_profiles:
        mask = profiles_arr == pid
        ax.scatter(
            X_pca2d[mask, 0], X_pca2d[mask, 1],
            c=[profile_color[pid]], label=str(pid),
            alpha=0.55, s=18, edgecolors="none"
        )
    ax.set_xlabel(f"PC1 ({var_exp[0]:.1f}% var)")
    ax.set_ylabel(f"PC2 ({var_exp[1]:.1f}% var)")
    ax.set_title("2D PCA Projection - coloured by Profile_ID")
    ax.legend(title="Profile_ID", bbox_to_anchor=(1.02, 1),
              loc="upper left", fontsize=7, markerscale=1.5)
    plt.tight_layout()
    p_pca2d = f"{output_path}/_pca2d_profiles.png"
    fig.savefig(p_pca2d, dpi=150)
    plt.close(fig)
    plot_paths.append(("2D PCA Projection by Profile_ID", p_pca2d))

    # --- 15c. Numeric mean by profile ---
    plot_num_cols = [
        c for c in raw_num_cols
        if c in df_stats.columns and c not in {"HP", "LP", "TRICT"}
    ]
    cmap_bar = cm.get_cmap("tab10", len(profile_order))

    for col in plot_num_cols[:10]:
        vals = [
            df_stats.loc[df_stats["Profile_ID"] == pid, col].values[0]
            if pid in df_stats["Profile_ID"].values else np.nan
            for pid in profile_order
        ]
        fig, ax = plt.subplots(figsize=(9, 4))
        ax.bar(profile_order, vals,
               color=[cmap_bar(i) for i in range(len(profile_order))])
        ax.set_title(f"Avg '{col}' by Profile")
        ax.set_xlabel("Profile_ID")
        ax.set_ylabel(f"Mean {col}")
        plt.xticks(rotation=30, ha="right")
        plt.tight_layout()
        p_num = f"{output_path}/_num_{col}.png"
        fig.savefig(p_num, dpi=150)
        plt.close(fig)
        plot_paths.append((f"Numeric: {col} by Profile", p_num))

    # --- 15d. Categorical distribution by profile ---
    for col in raw_cat_cols[:5]:
        ct = pd.crosstab(df["Profile_ID"], df[col], normalize="index")
        fig, ax = plt.subplots(figsize=(9, 4))
        ct.plot(kind="bar", stacked=True, ax=ax, colormap="tab20")
        ax.set_title(f"'{col}' distribution by Profile (row %)")
        ax.set_xlabel("Profile_ID")
        ax.set_ylabel("Proportion")
        ax.legend(title=col, bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=7)
        plt.xticks(rotation=30, ha="right")
        plt.tight_layout()
        p_cat = f"{output_path}/_cat_{col}.png"
        fig.savefig(p_cat, dpi=150)
        plt.close(fig)
        plot_paths.append((f"Categorical: {col} by Profile", p_cat))

    # =========================================================
    # 16. PDF REPORT
    # =========================================================
    pdf_path = f"{output_path}/InsightForge_Report.pdf"
    doc      = SimpleDocTemplate(
        pdf_path, pagesize=A4,
        leftMargin=2*rl_cm, rightMargin=2*rl_cm,
        topMargin=2*rl_cm,  bottomMargin=2*rl_cm
    )
    styles   = getSampleStyleSheet()
    h1, h2, body = styles["Heading1"], styles["Heading2"], styles["Normal"]
    story    = []

    # cover
    story.append(Spacer(1, 2*rl_cm))
    story.append(Paragraph("InsightForge - Analytical Report", h1))
    story.append(Spacer(1, 0.5*rl_cm))
    story.append(Paragraph(
        f"File: <b>{os.path.basename(file_path)}</b> &nbsp;|&nbsp; "
        f"Score column: <b>{target_score_col}</b> &nbsp;|&nbsp; "
        f"Threshold: <b>{threshold}th pct</b> &nbsp;|&nbsp; "
        f"DBSCAN eps: <b>{eps}</b>",
        body
    ))
    story.append(Spacer(1, 0.4*rl_cm))

    # profile summary table
    summary_data = [["Profile_ID", "N", "% Total", "HP Rate"]]
    for pid in profile_order:
        sub = df[df["Profile_ID"] == pid]
        summary_data.append([
            str(pid),
            str(len(sub)),
            f"{len(sub) / len(df) * 100:.1f}%",
            f"{sub['HP'].mean() * 100:.1f}%"
        ])

    tbl = Table(summary_data, hAlign="LEFT")
    tbl.setStyle(TableStyle([
        ("BACKGROUND",     (0, 0), (-1, 0),  colors.HexColor("#4C72B0")),
        ("TEXTCOLOR",      (0, 0), (-1, 0),  colors.white),
        ("FONTNAME",       (0, 0), (-1, 0),  "Helvetica-Bold"),
        ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.HexColor("#EEF2F8"), colors.white]),
        ("GRID",           (0, 0), (-1, -1), 0.5, colors.grey),
        ("FONTSIZE",       (0, 0), (-1, -1), 9),
        ("PADDING",        (0, 0), (-1, -1), 5),
    ]))
    story.append(tbl)
    story.append(PageBreak())

    # charts
    page_w = A4[0] - 4*rl_cm
    for title_txt, img_path in plot_paths:
        story.append(Paragraph(title_txt, h2))
        story.append(Spacer(1, 0.2*rl_cm))
        story.append(RLImage(img_path, width=page_w, height=page_w * 0.50))
        story.append(Spacer(1, 0.6*rl_cm))

    doc.build(story)

    # cleanup temp PNGs
    for _, img_path in plot_paths:
        try:
            os.remove(img_path)
        except OSError:
            pass

    print("PIPELINE COMPLETED SUCCESSFULLY")
    print(f"  -> Outputs saved in: {output_path}/")

    return {
        "shap_importance":  shap_output,
        "profile_stats":    df_stats,
        "profile_rules":    df_rules,
        "profile_features": df_importances,
        "oddsratio":        df_or,
        "id_with_profile":  df[[id_col, "Profile_ID", "HP", "LP"]],
        "pdf_report":       pdf_path,
    }


In [5]:
results = insightforge_dsmodule(
    file_path="IF_abt_input/ActionForge.csv",        # path al tuo CSV
    sep=";",                     # separatore colonne
    decimal=",",                 # separatore decimale
    id_col="IDCLI",                 # colonna identificativo univoco
    target_score_col="score",    # colonna con lo score del modello
    threshold=10,                # percentile per definire HP/LP (es. top/bottom 10%)
    min_profile_size=0.05,       # dimensione minima cluster (5% del totale)
    eps=0.25,                    # raggio neighbourhood DBSCAN
    output_path="output_IF"      # cartella dove salvare tutti gli output
)

# Accedere agli output in memoria:
results["shap_importance"]   # lista dizionari con feature e mean_impact
results["profile_stats"]     # DataFrame statistiche aggregate per profilo
results["profile_rules"]     # DataFrame regole decision tree
results["profile_features"]  # DataFrame importanze feature
results["oddsratio"]         # DataFrame odds ratio logistica multinomiale
results["id_with_profile"]   # DataFrame id → Profile_ID, HP, LP
results["pdf_report"]        # path al PDF generato

C:\Users\luigi\anaconda3\Lib\site-packages\sklearn\model_selection\_search.py:317: UserWarning: The total space of parameters 2 is smaller than n_iter=10. Running 2 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
C:\Users\luigi\anaconda3\Lib\site-packages\sklearn\model_selection\_search.py:317: UserWarning: The total space of parameters 2 is smaller than n_iter=10. Running 2 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
C:\Users\luigi\anaconda3\Lib\site-packages\shap\explainers\_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(
C:\Users\luigi\AppData\Local\Temp\ipykernel_2244\3070401791.py:432: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap_scatter    = cm.get_cmap("ta

PIPELINE COMPLETED SUCCESSFULLY
  -> Outputs saved in: output_IF/


'output_IF/InsightForge_Report.pdf'

In [32]:
if __name__ == "__main__":
    insightforge_dsmodule(
        file_path="ABT_INSIGHTFORGE.csv",
        sep=";",
        decimal=",",
        id_col="IDCLI",
        target_score_col="score",
        threshold=10,
        min_profile_size=0.1,
        output_path="output_ds"
    )

C:\Users\luigi\anaconda3\Lib\site-packages\sklearn\model_selection\_search.py:317: UserWarning: The total space of parameters 2 is smaller than n_iter=10. Running 2 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


[LightGBM] [Info] Number of positive: 40, number of negative: 359
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000278 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 538
[LightGBM] [Info] Number of data points in the train set: 399, number of used features: 32
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.100251 -> initscore=-2.194443
[LightGBM] [Info] Start training from score -2.194443
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


C:\Users\luigi\anaconda3\Lib\site-packages\sklearn\model_selection\_search.py:317: UserWarning: The total space of parameters 2 is smaller than n_iter=10. Running 2 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


[LightGBM] [Info] Number of positive: 40, number of negative: 359
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000450 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 538
[LightGBM] [Info] Number of data points in the train set: 399, number of used features: 32
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.100251 -> initscore=-2.194443
[LightGBM] [Info] Start training from score -2.194443
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


C:\Users\luigi\anaconda3\Lib\site-packages\shap\explainers\_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


In [46]:

    output_path="output_ds"    
    os.makedirs(output_path, exist_ok=True)
    id_col="IDCLI"
    target_score_col="score"
    threshold=12
    min_profile_size=0.06
    output_path="output_ds"
    
  
    # =========================
    # LOAD DATA
    # =========================
    df = pd.read_csv("ABT_INSIGHTFORGE.csv", sep=";", decimal=",").drop("TGT",axis=1)

    # =========================
    # HP / LP CREATION
    # =========================
    p_low, p_high = np.percentile(df[target_score_col], [threshold, 100 - threshold])

    df["HP"] = np.where(df[target_score_col] >= p_high, 1, 0)
    df["LP"] = np.where(df[target_score_col] <= p_low, 1, 0)
    
    df["TRICT"] = np.where(
            df[target_score_col] <= p_low, 0,
            np.where(df[target_score_col] > p_high, 2, 1)
        )

    # =========================
    # PREPROCESSING
    # =========================
    X = df.drop([id_col, target_score_col], axis=1, errors="ignore")

    cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
    num_cols = X.select_dtypes(exclude=["object", "category"]).columns.tolist()

    ohe = OneHotEncoder(sparse_output=False, drop="first", handle_unknown="ignore")

    if len(cat_cols) > 0:
        X_cat = ohe.fit_transform(X[cat_cols])
        df_cat = pd.DataFrame(X_cat, columns=ohe.get_feature_names_out(cat_cols), index=df.index)
    else:
        df_cat = pd.DataFrame()

    X = pd.concat([X[num_cols], df_cat], axis=1)

    X = X.drop(["HP", "LP", "TRICT"], axis=1, errors="ignore")
    X = X.fillna(X.mean())

    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)

    # =========================
    # SHAP MODEL
    # =========================
    Y = df["HP"].values

    gb = lgb.LGBMClassifier()
    rf = RandomForestClassifier()

    search_gb = RandomizedSearchCV(gb, {"n_estimators":[50,100]}, cv=3, scoring="roc_auc", n_jobs=-1)
    search_rf = RandomizedSearchCV(rf, {"n_estimators":[50,100]}, cv=3, scoring="roc_auc", n_jobs=-1)

    search_gb.fit(X_scaled, Y)
    search_rf.fit(X_scaled, Y)

    model = (search_gb if search_gb.best_score_ > search_rf.best_score_ else search_rf).best_estimator_
    model.fit(X_scaled, Y)
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_scaled)
    
    # =========================
    # ROBUST SHAP HANDLING
    # =========================
    
    if isinstance(shap_values, list):
        # caso classico: lista di array (una per classe)
        shap_array = shap_values[1] if len(shap_values) > 1 else shap_values[0]
    
    else:
        shap_array = shap_values
    
    # Se è 3D (succede con alcuni modelli)
    if len(shap_array.shape) == 3:
        # (samples, features, classes) → prendiamo classe positiva
        shap_array = shap_array[:, :, 1]
    
    # Ora garantito: (n_samples, n_features)
    mean_impact = np.abs(shap_array).mean(axis=0)
    
    # Safety: forza a 1D
    mean_impact = np.array(mean_impact).reshape(-1)
    
    # =========================
    # BUILD OUTPUT
    # =========================
    shap_output = []
    
    for i, feat in enumerate(X.columns):
        val = mean_impact[i]
    
        # ulteriore sicurezza (mai più crash)
        if isinstance(val, (np.ndarray, list)):
            val = np.mean(val)
    
        shap_output.append({
            "feature": feat,
            "mean_impact": float(val)
        })
    
    # sort
    shap_output = sorted(shap_output, key=lambda x: x["mean_impact"], reverse=True)
    
    # export
    with open(f"{output_path}/shap_importance.json", "w") as f:
        json.dump(shap_output, f, indent=2)



C:\Users\luigi\anaconda3\Lib\site-packages\sklearn\model_selection\_search.py:317: UserWarning: The total space of parameters 2 is smaller than n_iter=10. Running 2 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


[LightGBM] [Info] Number of positive: 48, number of negative: 351
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000411 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 536
[LightGBM] [Info] Number of data points in the train set: 399, number of used features: 31
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.120301 -> initscore=-1.989585
[LightGBM] [Info] Start training from score -1.989585
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


C:\Users\luigi\anaconda3\Lib\site-packages\sklearn\model_selection\_search.py:317: UserWarning: The total space of parameters 2 is smaller than n_iter=10. Running 2 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


[LightGBM] [Info] Number of positive: 48, number of negative: 351
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000110 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 536
[LightGBM] [Info] Number of data points in the train set: 399, number of used features: 31
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.120301 -> initscore=-1.989585
[LightGBM] [Info] Start training from score -1.989585
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


C:\Users\luigi\anaconda3\Lib\site-packages\shap\explainers\_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


In [53]:


    
  # =========================
# PCA + DBSCAN
 # =========================
pca = PCA(n_components=0.71)
X_pca = pca.fit_transform(X_scaled)

min_samples = max(2, int(min_profile_size * len(X_pca)))

db = DBSCAN(eps=0.25, min_samples=min_samples, metric="cosine")
df["cluster"] = db.fit_predict(X_pca)

df["base_propensity"] = df["HP"].mean()
df["cluster_propensity"] = df.groupby("cluster")["HP"].transform("mean")

df["final_cluster"] = np.where(
        df["cluster_propensity"] / df["base_propensity"] > 1.2,
        df["cluster"],
        "NotHP"
    )

df.info()   
df.drop("cluster",axis=1)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 399 entries, 0 to 398
Data columns (total 30 columns):
 #   Column                                            Non-Null Count  Dtype  
---  ------                                            --------------  -----  
 0   IDCLI                                             399 non-null    int64  
 1   Professione                                       399 non-null    object 
 2   Capacita_d_ spsa                                  399 non-null    int64  
 3   LTV                                               399 non-null    int64  
 4   RFM_Score                                         399 non-null    int64  
 5   Numero di Prodtti Aquistati negli ultimi 12 mesi  399 non-null    int64  
 6   Eta                                               399 non-null    int64  
 7   Tenure                                            399 non-null    int64  
 8   LivelloDigitalizzazione                           399 non-null    int64  
 9   GrandCentr           

,IDCLI,Professione,Capacita_d_ spsa,LTV,RFM_Score,Numero di Prodtti Aquistati negli ultimi 12 mesi,Eta,Tenure,LivelloDigitalizzazione,GrandCentr,...,TRICT,cluster_x,base_propensity,cluster_propensity,final_cluster,cluster_y,p_value_x,z_stat_x,p_value_y,z_stat_y
0,1,Professionista,4,3,7,3,50,10,5,1,...,2,-1,0.120301,0.214286,-1,-1,3.120898e-04,3.605053,3.120898e-04,3.605053
1,2,Imprenditore,3,4,9,3,38,9,3,1,...,1,-1,0.120301,0.214286,-1,-1,3.120898e-04,3.605053,3.120898e-04,3.605053
2,3,Impiegato,3,3,6,3,31,5,3,1,...,2,-1,0.120301,0.214286,-1,-1,3.120898e-04,3.605053,3.120898e-04,3.605053
3,4,Professionista,4,4,6,4,37,15,4,1,...,1,-1,0.120301,0.214286,-1,-1,3.120898e-04,3.605053,3.120898e-04,3.605053
4,5,Imprenditore,3,4,7,3,45,10,4,1,...,1,2,0.120301,0.054726,NotHP,NotHP,1.940816e-08,-5.617196,1.940816e-08,-5.617196
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
394,395,Tassista,1,2,1,1,61,6,3,0,...,0,2,0.120301,0.054726,NotHP,NotHP,1.940816e-08,-5.617196,1.940816e-08,-5.617196
395,396,Pensionato,2,2,1,1,74,6,2,0,...,1,2,0.120301,0.054726,NotHP,NotHP,1.940816e-08,-5.617196,1.940816e-08,-5.617196
396,397,Pensionato,2,2,3,1,71,7,2,0,...,1,2,0.120301,0.054726,NotHP,NotHP,1.940816e-08,-5.617196,1.940816e-08,-5.617196
397,398,Tassista,1,2,4,1,59,4,3,0,...,0,2,0.120301,0.054726,NotHP,NotHP,1.940816e-08,-5.617196,1.940816e-08,-5.617196


In [54]:
# =========================
    # STAT TEST
    # =========================
results = []

for cl in df["final_cluster"].unique():
    df_cl = df[df["final_cluster"] == cl]
    df_rest = df[df["final_cluster"] != cl]
    
    success = [df_cl["HP"].sum(), df_rest["HP"].sum()]
    n = [len(df_cl), len(df_rest)]
    
    stat, p = proportions_ztest(success, n)
    
    results.append({"cluster": cl, "p_value": p, "z_stat": stat})
    
df = df.merge(pd.DataFrame(results), left_on="final_cluster", right_on="cluster", how="left")
    
df["final_cluster"] = np.where(
            (df["p_value"] <= 0.05) & (np.abs(df["z_stat"]) >= 1.96),
            df["final_cluster"],
            "NotHP"
        )


MergeError: Passing 'suffixes' which cause duplicate columns {'cluster_x'} is not allowed.

In [ ]:

    # =========================
    # DECISION TREE
    # =========================
    drop_cols = [
        id_col, target_score_col, "HP", "LP", "TRICT",
        "cluster", "cluster_propensity", "base_propensity","cluster_y","cluster_x"
        "final_cluster", "p_value", "z_stat", "cluster_y","cluster_x"
    ]

    X_dt = df.drop(drop_cols, axis=1, errors="ignore")

    cat_cols = X_dt.select_dtypes(include=["object"]).columns.tolist()
    num_cols = X_dt.select_dtypes(exclude=["object"]).columns.tolist()

    if len(cat_cols) > 0:
        X_cat = ohe.fit_transform(X_dt[cat_cols])
        df_cat = pd.DataFrame(X_cat, columns=ohe.get_feature_names_out(cat_cols), index=df.index)
    else:
        df_cat = pd.DataFrame()

    X_dt = pd.concat([X_dt[num_cols], df_cat], axis=1)

    feature_names = list(X_dt.columns)

    X_dt = X_dt.fillna(X_dt.mean()).values
    Y_dt = df["final_cluster"].values

    dt = DecisionTreeClassifier()
    dt.fit(X_dt, Y_dt)

    df["leaf_id"] = dt.apply(X_dt)
    df["leaf_propensity"] = df.groupby("leaf_id")["HP"].transform("mean")

    df["Profile_ID"] = np.where(
        df["leaf_propensity"] / df["base_propensity"] > 1.2,
        df["leaf_id"],
        "Generic"
    )

    # =========================
    # RULE EXTRACTION
    # =========================
    def extract_rules(tree, feature_names):
        children_left = tree.tree_.children_left
        children_right = tree.tree_.children_right
        features = tree.tree_.feature
        thresholds = tree.tree_.threshold

        rules = {}

        def recurse(node, path):
            if children_left[node] == children_right[node]:
                rules[node] = " AND ".join(path)
            else:
                feat = feature_names[features[node]]
                thr = thresholds[node]

                recurse(children_left[node], path + [f"{feat} <= {thr:.3f}"])
                recurse(children_right[node], path + [f"{feat} > {thr:.3f}"])

        recurse(0, [])
        return rules

    rules_dict = extract_rules(dt, feature_names)

    df_rules = pd.DataFrame({
        "leaf_id": df["leaf_id"],
        "rule": df["leaf_id"].map(rules_dict)
    }).drop_duplicates()

    df_rules = df_rules.merge(
        df[["leaf_id", "Profile_ID"]].drop_duplicates(),
        on="leaf_id",
        how="left"
    )

    df_rules.to_csv(f"{output_path}/PROFILE_RULES.csv", index=False)

    # =========================
    # FEATURE IMPORTANCE (DECISION TREE)
    # =========================

    importances = dt.feature_importances_

    df_importances = pd.DataFrame({
        "feature": feature_names,
        "importance": importances
    })

    # Keep only non-zero importance (same logic as original code)
    df_importances = df_importances[df_importances["importance"] > 0]

    # Sort descending
    df_importances = df_importances.sort_values(by="importance", ascending=False)

    # Export
    df_importances.to_csv(f"{output_path}/PROFILE_FEATURES.csv", index=False)
    
    # =========================
    # PROFILE STATS
    # =========================
    df["profile_volume"] = df.groupby("Profile_ID")[id_col].transform("count")
    df["profile_pct"] = df["profile_volume"] / len(df)

    numeric_cols = df.select_dtypes(include="number").columns.tolist()
    numeric_cols = [c for c in numeric_cols if c not in ["Profile_ID"]]

    agg_dict = {col: "mean" for col in numeric_cols}

    df_stats = df.groupby("Profile_ID").agg(agg_dict).reset_index()

    df_stats.to_csv(f"{output_path}/PROFILE_STATS.csv", index=False)

    # =========================
    # EXPORT ID PROFILE
    # =========================
    df[[id_col, "Profile_ID", "HP", "LP","cluster_y","cluster_x"]].to_csv(
        f"{output_path}/ID_W_PROFILE.csv", index=False
    )

    # =========================
    # LOGISTIC REGRESSION
    # =========================

    X_lr = df.drop([id_col, "Profile_ID", "HP" ,"LP","cluster_y","cluster_x"], axis=1, errors="ignore")
    
    # Separate numeric and categorical
    num_cols = X_lr.select_dtypes(include=["number"]).columns.tolist()
    cat_cols = X_lr.select_dtypes(exclude=["number"]).columns.tolist()
    
    # Impute missing values
    if len(num_cols) > 0:
        imputer_num = SimpleImputer(strategy="mean")
        X_num = imputer_num.fit_transform(X_lr[num_cols])
    else:
        X_num = np.empty((len(X_lr), 0))
    
    if len(cat_cols) > 0:
        X_cat_raw = X_lr[cat_cols].fillna("missing")
        ohe_lr = OneHotEncoder(sparse_output=False, drop="first", handle_unknown="ignore")
        X_cat = ohe_lr.fit_transform(X_cat_raw)
    else:
        X_cat = np.empty((len(X_lr), 0))
    
    # Combine numeric + categorical
    X_lr_final = np.hstack([X_num, X_cat])
    
    # Scale
    scaler = MinMaxScaler()
    X_lr_final = scaler.fit_transform(X_lr_final)
    
    # Target
    Y_lr = df["Profile_ID"].values
    
    # Fit Logistic Regression
    log = LogisticRegression(max_iter=5000)
    log.fit(X_lr_final, Y_lr)
    
    # Odds ratio
    odds = np.exp(log.coef_)
    df_or = pd.DataFrame(odds.T)
    df_or.to_csv(f"{output_path}/oddsratio.csv", index=False)